# 项目：评估和清理英国电商公司销售数据

## **分析目标**

此数据分析的目的是，根据市场销售数据，挖掘畅销产品，以便制定更有效的市场策略来提升营收。

本实战项目的目的在于练习评估数据干净和整洁度，并且基于评估结果，对数据进行清洗，从而得到可供下一步分析的数据。

## **简介**

原始数据集记录了一家英国在线零售公司在2010年12月1日至2011年12月9日期间的所有交易情况，涵盖了该公司在全球不同国家和地区的业务数据。该公司主要销售覆盖各个场景的礼品，包括但不限于生日礼品、结婚纪念品、圣诞礼品等等。该公司的客户群体主要包括批发商和个人消费者，其中批发商占据了相当大的比例。

数据每列的含义如下：
- `InvoiceNo`: 发票号码。6位数，作为交易的唯一标识符。如果这个代码以字母“c”开头，表示这笔交易被取消。
- `StockCode`: 产品代码。5位数，作为产品的唯一标识符。
- `Description`: 产品名称。
- `Quantity`: 产品在交易中的数量。
- `InvoiceDate`: 发票日期和时间。交易发生的日期和时间。
- `UnitPrice`: 单价。价格单位为英镑（£）。
- `CustomerID`: 客户编号。5位数，作为客户的唯一标识符。
- `Country`: 国家名称。客户所居住的国家的名称。

## **读取数据**

##### 导入数据分析所需要库, 并通过pandas的`read_csv`函数, 将原始数据文件"e_commerce.csv"里的数据内容, 解析为DataFrame, 并赋值给变量`original_data`

In [1]:
import pandas as pd
original_data = pd.read_csv("e_commerce.csv")

In [2]:
# 创建源文件的副本, 后续的清理是对副本进行操作的
cleaned_data = original_data.copy()
original_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## **评估与清理数据**

在这一部分, 我将对在上一部分建立的`original_data`这个DataFrame所包含的数据进行评估.

评估主要从两个方面进行: 结构和内容, 即整齐度和干净度. 数据的结构性问题指的是数据不同时符合"每列是一个变量, 每行是一个观察值,每个单元格是一个值"这三个标准, 数据的内容性问题包括存在丢失数据, 重复数据, 无效数据等问题

### **评估数据整齐度**

In [3]:
original_data.sample(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
150331,549425,21069,VINTAGE BILLBOARD TEA MUG,6,4/8/2011 11:54,1.25,14030.0,United Kingdom
515996,579840,22661,CHARLOTTE BAG DOLLY GIRL DESIGN,10,11/30/2011 16:19,0.85,18158.0,United Kingdom
171370,551414,23182,TOILET SIGN OCCUPIED OR VACANT,24,4/28/2011 13:35,0.83,15622.0,United Kingdom
5979,536876,22862,LOVE HEART NAPKIN BOX,1,12/3/2010 11:36,8.47,NaN,United Kingdom
32969,539213,22661,CHARLOTTE BAG DOLLY GIRL DESIGN,10,12/16/2010 12:23,0.85,12877.0,United Kingdom
461665,575947,22456,NATURAL SLATE CHALKBOARD LARGE,1,11/13/2011 11:50,9.96,NaN,United Kingdom
458418,575837,20971,PINK BLUE FELT CRAFT TRINKET BOX,6,11/11/2011 11:37,1.25,12748.0,United Kingdom
415703,572549,22784,LANTERN CREAM GAZEBO,1,10/24/2011 17:03,10.79,NaN,United Kingdom
463124,576053,22951,60 CAKE CASES DOLLY GIRL DESIGN,1,11/13/2011 14:53,0.55,16726.0,United Kingdom
146953,549023,85094,CANDY SPOT EGG WARMER RABBIT,3,4/5/2011 16:27,0.83,NaN,United Kingdom


从多次抽样的十行数据来看, 数据符合"每列是一个变量, 每行是一个观察值, 每个单元格是一个值"的整洁性原则, 具体来看每行是关于某商品的一次交易, 每列是交易相关的各个变量,因此不存在结构性问题

### **评估数据干净度**

**初步评估数据干净度**

使用info了解数据大致信息

In [27]:
original_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


从输出结果来看, 数据一共有541909条观察值.    
其中`Description`和`CustomerID`两个变量的数值存在缺失值, 需要做进一步评估.   
此外, 有两处应该做数据格式转换: `InvoiceData`的数据类型应该转换为日期类型, `CustomerID`应该转换为字符串类型.

把`InvoiceDate`变量的数据类型转换为datetime类型

In [5]:
cleaned_data["InvoiceDate"] = pd.to_datetime(cleaned_data["InvoiceDate"])
cleaned_data["InvoiceDate"]

0        2010-12-01 08:26:00
1        2010-12-01 08:26:00
2        2010-12-01 08:26:00
3        2010-12-01 08:26:00
4        2010-12-01 08:26:00
                 ...        
541904   2011-12-09 12:50:00
541905   2011-12-09 12:50:00
541906   2011-12-09 12:50:00
541907   2011-12-09 12:50:00
541908   2011-12-09 12:50:00
Name: InvoiceDate, Length: 541909, dtype: datetime64[ns]

把`CustomerID`变量的数据类型转换为字符串:

In [6]:
cleaned_data["CustomerID"] = cleaned_data["CustomerID"].astype(str)
cleaned_data["CustomerID"]

0         17850.0
1         17850.0
2         17850.0
3         17850.0
4         17850.0
           ...   
541904    12680.0
541905    12680.0
541906    12680.0
541907    12680.0
541908    12680.0
Name: CustomerID, Length: 541909, dtype: object

把`CustomerID`变量值结尾的`".0"`删除

In [7]:
cleaned_data["CustomerID"].str.slice(0, -2)

0         17850
1         17850
2         17850
3         17850
4         17850
          ...  
541904    12680
541905    12680
541906    12680
541907    12680
541908    12680
Name: CustomerID, Length: 541909, dtype: object

**评估数据干净度1: 缺失值**

了解到`Description`存在缺失值后, 调出所有缺失`Description`变量值的观察值

In [8]:
original_data[original_data["Description"].isnull()]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/2010 11:52,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,12/1/2010 14:32,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,12/1/2010 14:33,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,12/1/2010 14:33,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,12/1/2010 14:34,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,12/7/2011 18:26,0.0,NaN,United Kingdom
535326,581203,23406,NaN,15,12/7/2011 18:31,0.0,NaN,United Kingdom
535332,581209,21620,NaN,6,12/7/2011 18:35,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,12/8/2011 10:33,0.0,NaN,United Kingdom


如上, 有1454条交易数据都缺失Description变量值

观察输出结果, 这些缺失`Description`的交易数据, `Unitprice`都为0.    

为了验证猜想, 我们增加筛选条件来查看是否存在`Description`变量缺失而`Unitprice`不为0的数据

In [9]:
original_data[original_data["Description"].isnull() & original_data["UnitPrice"] != 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


筛选结果为空, 说明该数据集中缺失`Description`的观察值, 同时也不具备有效的`UnitPrice`值.  


本次数据分析的目的是筛选出畅销的商品, `Description`和`UnitPrice`都是后续商品交易分析的重要变量.   
如果他们同时缺失, 对应的数据对于后续分析就没有实际意义了, 所以选择删除掉这些观察值

删除`Description`变量值缺失的观察值, 并检查删除后的`Description`变量值的缺失值情况

In [10]:
cleaned_data.dropna(subset=["Description"], inplace=True)
cleaned_data["Description"].isnull().sum()

np.int64(0)

##### 继续根据条件筛选出缺失`CustomerID`变量值的观察值

In [11]:
original_data[original_data["CustomerID"].isnull()]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/2010 11:52,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,12/1/2010 14:32,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/1/2010 14:32,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,12/1/2010 14:32,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,12/1/2010 14:32,1.66,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,12/9/2011 10:26,4.13,NaN,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,12/9/2011 10:26,4.13,NaN,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,12/9/2011 10:26,4.96,NaN,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,12/9/2011 10:26,10.79,NaN,United Kingdom


如上, 一共有135080条交易数据缺失了`CustomerID`这个变量值

可以发现缺失`CustomerID`变量值的观察值, 对应的`Description`和`UnitPrice`值并不是都缺失, 也就是说`CustomerID`变量值的缺失并不影响后续的数据分析, 所以保留这些数据

**评估数据干净度2: 重复值**

根据数据变量含义来看, 数据集的有三个唯一标识符变量: `InvoiceNo`, `StockCode`, `CustomerID`. 但是在一次交易中可能包含多件商品, 所以描述观察值的时候`InvoiceNo` 可以重复; 而不同的交易中可以出现相同的产品, 所以`StockCode`可以重复; 同时, 相同顾客也可以进行多次购买, 所以描述观察值时`CustomerID`也可以重复

但是需要注意所有变量同时重复的情况, 因为这代表着订单信息的重复提交, 需要删除重复值, 只保留一条观察值

In [12]:
cleaned_data.duplicated().sum()

np.int64(5268)

如上, 数据中确实存在5268条交易数据是所有变量同时重复的情况, 这会影响我们对商品销量的判断 

所以删除这些观察值, 并检查删除后DataFrame的行重复情况

In [13]:
cleaned_data.drop_duplicates(inplace=True)
cleaned_data.duplicated().sum()

np.int64(0)

**评估数据干净度3: 不一致数据**

排除3个唯一标识符, 再排除数值型和datetime这些固定格式的数据, 还剩`Country`变量需要进一步观察是否存在不同值指代同一个国家的情况

In [14]:
cleaned_data["Country"].value_counts()

Country
United Kingdom          488639
Germany                   9480
France                    8541
EIRE                      8184
Spain                     2528
Netherlands               2371
Belgium                   2069
Switzerland               1994
Portugal                  1510
Australia                 1258
Norway                    1086
Italy                      803
Channel Islands            757
Finland                    695
Cyprus                     611
Sweden                     461
Unspecified                442
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     294
China                      284
Singapore                  229
USA                        218
UK                         206
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United States               73
United Arab Emirates        68


根据输出结果, `"USA"`和`"United States"`同时指代美国; `"U.K."`, `"UK"`和`"United Kingdom"`同时指代英国. 

对两个不一致指代的对象进行替换, 并检查替换后`USA`, `UK`, `U.K.`的变量值个数

In [28]:
cleaned_data.replace({'USA': 'United States', 'U.K.': 'United Kingdom', 'UK': 'United Kingdom'}, inplace=True)

Country
United Kingdom          480549
Germany                   9027
France                    8393
EIRE                      7883
Spain                     2480
Netherlands               2363
Belgium                   2031
Switzerland               1959
Portugal                  1492
Australia                 1184
Norway                    1072
Italy                      758
Channel Islands            747
Finland                    685
Cyprus                     603
Sweden                     450
Unspecified                442
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     292
China                      280
Singapore                  222
Iceland                    182
United States              179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         58


In [31]:
print(len(cleaned_data[cleaned_data["Country"] == 'USA']))
print(len(cleaned_data[cleaned_data["Country"] == 'UK']))
print(len(cleaned_data[cleaned_data["Country"] == 'U.K.']))

0
0
0


**评估数据干净度4: 无效/错误数据**

先使用describe方法, 对数值统计信息进行快速了解

In [16]:
cleaned_data.describe()

,Quantity,InvoiceDate,UnitPrice
count,535187.000000,535187,535187.000000
mean,9.671593,2011-07-04 11:43:44.485273600,4.645242
min,-80995.000000,2010-12-01 08:26:00,-11062.060000
25%,1.000000,2011-03-28 11:34:00,1.250000
50%,3.000000,2011-07-19 15:38:00,2.080000
75%,10.000000,2011-10-19 08:20:00,4.130000
max,80995.000000,2011-12-09 12:50:00,38970.000000
std,219.059056,NaN,97.364810


观察输出结果, `Quantity`和`UnitPrice`都出现了负数, 这会对后续数值分析产生影响 

因此, 我们先筛选出`Quantity`为负数的观察值

In [17]:
cleaned_data[(cleaned_data["Quantity"] < 0 )]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


根据筛选结果, `Quantity`为负的观察值,`InvoiceNo`似乎都以C开头, 这代表交易被取消  
为了验证猜想, 我们增加筛选条件, 看看是否存在`Quantity`为负且`InvoiceNo`不以C开头的观察值

In [18]:
cleaned_data[(cleaned_data["Quantity"] < 0) & (cleaned_data["InvoiceNo"].str[0] != "C")]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
7313,537032,21275,?,-30,2010-12-03 16:50:00,0.0,nan,United Kingdom
13217,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,nan,United Kingdom
13218,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,nan,United Kingdom
13264,537432,35833G,damages,-43,2010-12-06 16:10:00,0.0,nan,United Kingdom
21338,538072,22423,faulty,-13,2010-12-09 14:10:00,0.0,nan,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,2011-12-07 18:36:00,0.0,nan,United Kingdom
535335,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,nan,United Kingdom
535336,581213,22576,check,-30,2011-12-07 18:38:00,0.0,nan,United Kingdom
536908,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,nan,United Kingdom


进一步的筛查结果显示, 仍然有474条观察值不满足以上猜想, 以上猜想错误 


但观察筛查结果发现, 这些观察值的`UnitPrice`都为0, 进一步增加`UnitPrice`条件进行验证

In [19]:
cleaned_data[(cleaned_data["Quantity"] < 0) & (cleaned_data["InvoiceNo"].str[0] != "C") & (cleaned_data["UnitPrice"] != 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


根据筛查结果, 猜想得到验证, 当`Quantity`为负时, 观察值满足以下条件之一:    
- `InvoiceVo`以C开头, 表示订单被取消  
- `UnitPrice`为0, 表示商品单价为0英镑

  
这些数据均不是有效销售数据, 对后续数值分析没有实际作用, 一次将`Quantity`为负的观察值都删除

按条件筛选观察值来对DataFrame重新赋值, 并检查重新赋值后Quantity变量值小于0的观察值个数

In [32]:
cleaned_data = cleaned_data[cleaned_data["Quantity"] >= 0]
len(cleaned_data[cleaned_data["Quantity"] < 0])

0

接下来筛选出`UnitPrice`为负的观察值

In [21]:
cleaned_data[cleaned_data["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,nan,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,nan,United Kingdom


根据输出结果, `UnitPrice`为负的观察值都是坏账调整, 不是实际交易信息, 可直接删除

删除`UnitPrice`为负的观察值, 并检查删除后`UnitPrice`小于0的观察值个数

In [22]:
cleaned_data = cleaned_data[cleaned_data["UnitPrice"] >= 0]

In [23]:
len(cleaned_data[cleaned_data["UnitPrice"] < 0])

0

## **数据清理步骤小结**

这一步已经模拟实际的数据预处理, 合并到数据评估过程中, 在此仅小结一下执行了的清理步骤

根据前面评估部分得到的结论, 我们需要进行的数据清理包括:  
- 把`InvoiceDate`变量的数据类型转换为为日期时间
- 把`CustomerID`变量的数据类型转换为字符串
- 把`Description`变量缺失的观察值删除
- 把`Country`变量值`"USA"`替换为`"United States"`
- 把`Country`变量值`"UK"`、`"U.K."`替换为`"United Kingdom"`
- 把`Quantity`变量值为负数的观察值删除
- 把`UnitPrice`变量值为负数的观察值删除


为了区分开经过清理的数据和原始的数据, 我们创建新变量`cleaned_data`, 让它被赋值为`origina_data`复制出的副本, 我们所有的清理步骤都使运用在`cleanen_data`上的 

## **保存清理后的数据**

完成数据清理后, 把干净整齐的数据保存到新的文件里, 文件名为`e_commerce_cleaned.csv`

In [24]:
cleaned_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


可以看到该数据集的索引并无实际含义, 因此在数据保存时选择不保存索引

In [25]:
cleaned_data.to_csv('e_commerce_cleaned.csv', index=False)

读取保存了的新文件名以检查保存情况

In [26]:
pd.read_csv("e_commerce_cleaned.csv").head()

C:\Users\面面\AppData\Local\Temp\ipykernel_13380\2686214642.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("e_commerce_cleaned.csv").head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
